In [22]:
import pandas as pd
import numpy as np

In [23]:
registry = pd.read_csv("../data/university_raw_data/university_registry.csv")
qs = pd.read_csv("../data/university_raw_data/qs_master_clean.csv")
the = pd.read_csv("../data/university_raw_data/the_master.csv")
openalex = pd.read_csv("../data/university_raw_data/openalex_master.csv")
wb = pd.read_csv("../data/university_raw_data/worldbank_indicators_raw.csv")

In [24]:
print("Registry:", registry.shape)
print("QS:", qs.shape)
print("THE:", the.shape)
print("OpenAlex:", openalex.shape)
print("World Bank:", wb.shape)

Registry: (3565, 4)
QS: (7980, 15)
THE: (10051, 28)
OpenAlex: (2865, 12)
World Bank: (1250, 10)


In [25]:
def clean_columns(df):
    df.columns = (
        df.columns.str.strip()
                  .str.lower()
                  .str.replace(" ", "_", regex=False)
                  .str.replace("-", "_", regex=False)
    )
    return df

registry = clean_columns(registry)
qs = clean_columns(qs)
the = clean_columns(the)
openalex = clean_columns(openalex)
wb = clean_columns(wb)

In [26]:
print(registry.columns.tolist())
print(qs.columns.tolist())
print(the.columns.tolist())
print(openalex.columns.tolist())
print(wb.columns.tolist())

['university_id', 'normalized_name', 'university_name', 'country']
['university_id', 'university', 'country', 'year', 'qs_rank', 'qs_score', 'academic_reputation_score', 'employer_reputation_score', 'faculty_student_score', 'citations_per_faculty_score', 'international_faculty_score', 'international_students_score', 'international_research_network_score', 'employment_outcomes_score', 'sustainability_score']
['rank_order', 'rank', 'name', 'scores_overall', 'scores_overall_rank', 'scores_teaching', 'scores_teaching_rank', 'scores_research', 'scores_research_rank', 'scores_citations', 'scores_citations_rank', 'scores_industry_income', 'scores_industry_income_rank', 'scores_international_outlook', 'scores_international_outlook_rank', 'location', 'stats_number_students', 'stats_student_staff_ratio', 'stats_pc_intl_students', 'stats_female_male_ratio', 'aliases', 'subjects_offered', 'closed', 'unaccredited', 'normalized_name', 'year', 'university_id', 'stats_proportion_of_isr']
['university_

In [27]:
registry = registry.drop_duplicates(subset=["university_id"])

if "year" in qs.columns:
    qs = qs.drop_duplicates(subset=["university_id", "year"])

if "year" in the.columns:
    the = the.drop_duplicates(subset=["university_id", "year"])

openalex = openalex.drop_duplicates(subset=["university_id"])

In [28]:
registry = registry.drop_duplicates(subset=["university_id"])

if "year" in qs.columns:
    qs = qs.drop_duplicates(subset=["university_id", "year"])

if "year" in the.columns:
    the = the.drop_duplicates(subset=["university_id", "year"])

openalex = openalex.drop_duplicates(subset=["university_id"])

In [29]:
for df in [qs, the, wb]:
    if "year" in df.columns:
        df["year"] = pd.to_numeric(df["year"], errors="coerce")

In [30]:
country_map = {
    "USA": "United States",
    "U.S.A.": "United States",
    "United States of America": "United States",
    "UK": "United Kingdom",
    "U.K.": "United Kingdom",
    "Korea, Rep.": "South Korea",
    "Russian Federation": "Russia",
    "Viet Nam": "Vietnam",
    "Iran, Islamic Rep.": "Iran",
    "Hong Kong SAR": "Hong Kong",
    "Czech Republic": "Czechia"
}

if "country" in registry.columns:
    registry["country"] = registry["country"].replace(country_map)

if "country_name" in wb.columns:
    wb["country_name"] = wb["country_name"].replace(country_map)

In [31]:
duplicate_cols = [
    "university_name",
    "country",
    "normalized_name"
]

openalex = openalex.drop(
    columns=[c for c in duplicate_cols if c in openalex.columns],
    errors="ignore"
)

In [32]:
master = pd.merge(
    registry,
    qs,
    on="university_id",
    how="left",
    suffixes=("", "_qs")
)

print(master.shape)

(9969, 18)


In [33]:
master = pd.merge(
    master,
    the,
    on=["university_id", "year"],
    how="left",
    suffixes=("", "_the")
)

print(master.shape)

(9969, 44)


In [34]:
master = pd.merge(
    master,
    openalex,
    on="university_id",
    how="left"
)

print(master.shape)

(9969, 53)


In [35]:
master = master.rename(columns={"country": "country_name"})

master = pd.merge(
    master,
    wb,
    on=["country_name", "year"],
    how="left",
    suffixes=("", "_wb")
)

print(master.shape)

(9969, 61)


In [36]:
master = master.loc[:, ~master.columns.duplicated()]

In [37]:
master.replace(
    ["-", "--", "NA", "N/A", "null", "NULL", ""],
    np.nan,
    inplace=True
)

,university_id,normalized_name,university_name,country_name,university,country_qs,year,qs_rank,qs_score,academic_reputation_score,...,matched_name,score,country_code,gdp_per_capita_usd,education_expenditure_pct_gdp,tertiary_enrollment_ratio,population,literacy_rate_pct,data_source,download_timestamp_utc
0,U000001,aalborg university,Aalborg University,Denmark,Aalborg University,Denmark,2017.0,374,31.5,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,U000001,aalborg university,Aalborg University,Denmark,Aalborg University,Denmark,2018.0,379,32.2,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,U000001,aalborg university,Aalborg University,Denmark,Aalborg University,Denmark,2019.0,343,31.6,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,U000001,aalborg university,Aalborg University,Denmark,Aalborg University,Denmark,2020.0,324,33.0,NaN,...,NaN,NaN,DNK,60985.488560,7.38575,82.664330,5831404.0,NaN,World Bank API,2026-07-13 06:56:42 UTC
4,U000001,aalborg university,Aalborg University,Denmark,Aalborg University,Denmark,2021.0,305,34.0,NaN,...,NaN,NaN,DNK,69340.733492,6.95916,83.982292,5856733.0,NaN,World Bank API,2026-07-13 06:56:42 UTC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9964,U003561,zhytomyr polytechnic state university,Zhytomyr Polytechnic State University,Ukraine,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9965,U003562,ziane achour university of djelfa,Ziane Achour University of Djelfa,Algeria,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9966,U003563,ziauddin university,Ziauddin University,Pakistan,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9967,U003564,zonguldak bulent ecevit university,Zonguldak Bülent Ecevit University,Turkey,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [38]:
sort_cols = [c for c in ["university_id", "year"] if c in master.columns]

master = master.sort_values(sort_cols)

master.reset_index(drop=True, inplace=True)

In [39]:
print(master.shape)

master.head()

(9969, 61)


,university_id,normalized_name,university_name,country_name,university,country_qs,year,qs_rank,qs_score,academic_reputation_score,...,matched_name,score,country_code,gdp_per_capita_usd,education_expenditure_pct_gdp,tertiary_enrollment_ratio,population,literacy_rate_pct,data_source,download_timestamp_utc
0,U000001,aalborg university,Aalborg University,Denmark,Aalborg University,Denmark,2017.0,374,31.5,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,U000001,aalborg university,Aalborg University,Denmark,Aalborg University,Denmark,2018.0,379,32.2,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,U000001,aalborg university,Aalborg University,Denmark,Aalborg University,Denmark,2019.0,343,31.6,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,U000001,aalborg university,Aalborg University,Denmark,Aalborg University,Denmark,2020.0,324,33.0,NaN,...,NaN,NaN,DNK,60985.488560,7.38575,82.664330,5831404.0,NaN,World Bank API,2026-07-13 06:56:42 UTC
4,U000001,aalborg university,Aalborg University,Denmark,Aalborg University,Denmark,2021.0,305,34.0,NaN,...,NaN,NaN,DNK,69340.733492,6.95916,83.982292,5856733.0,NaN,World Bank API,2026-07-13 06:56:42 UTC


In [40]:
missing = master.isnull().sum().sort_values(ascending=False)

missing.head(20)

literacy_rate_pct                       9192
stats_proportion_of_isr                 9062
matched_name                            8958
score                                   8958
international_faculty_score             8598
sustainability_score                    8572
international_students_score            8552
citations_per_faculty_score             8496
faculty_student_score                   8496
employment_outcomes_score               8496
international_research_network_score    8476
employer_reputation_score               8473
academic_reputation_score               8472
education_expenditure_pct_gdp           7158
stats_female_male_ratio                 7045
scores_international_outlook            6919
scores_industry_income                  6919
scores_overall                          6919
scores_teaching                         6919
scores_citations                        6919
dtype: int64

In [41]:
master.to_csv("final_master_dataset.csv", index=False)

print("=" * 50)
print(" Merge Completed Successfully")
print("=" * 50)
print("Rows    :", master.shape[0])
print("Columns :", master.shape[1])
print("\nDataset saved as 'final_master_dataset.csv'")

 Merge Completed Successfully
Rows    : 9969
Columns : 61

Dataset saved as 'final_master_dataset.csv'


In [42]:
num_cols = master.select_dtypes(include=np.number).columns

for col in num_cols:
    master[col] = master[col].fillna(master[col].median())


cat_cols = master.select_dtypes(exclude=np.number).columns

for col in cat_cols:
    mode = master[col].mode()

    if len(mode):
        master[col] = master[col].fillna(mode[0])
    else:
        master[col] = master[col].fillna("Unknown")

print("Remaining Missing Values")
master.to_csv("final_master.csv", index=False)
print(master.isnull().sum().sum())

Remaining Missing Values
0


In [43]:
# Total number of missing values in the entire dataset
print("Total Missing Values:", master.isnull().sum().sum())

Total Missing Values: 0


In [44]:
missing_df = pd.DataFrame({
    "Missing Values": master.isnull().sum(),
    "Missing Percentage": (master.isnull().sum() / len(master)) * 100
})

missing_df = missing_df[missing_df["Missing Values"] > 0]

missing_df = missing_df.sort_values(
    by="Missing Percentage",
    ascending=False
)

missing_df

,Missing Values,Missing Percentage


In [45]:
master.describe().T

,count,mean,std,min,25%,50%,75%,max
year,9969.0,2.020401e+03,2.053924e+00,2017.000000,2.019000e+03,2.020000e+03,2.022000e+03,2.024000e+03
rank_order,9969.0,1.293695e+04,8.307234e+04,10.000000,5.700000e+03,5.700000e+03,5.700000e+03,1.000746e+06
scores_overall_rank,9969.0,1.293695e+04,8.307234e+04,10.000000,5.700000e+03,5.700000e+03,5.700000e+03,1.000746e+06
scores_teaching,9969.0,2.919206e+01,8.833213e+00,10.600000,2.790000e+01,2.790000e+01,2.790000e+01,9.900000e+01
scores_teaching_rank,9969.0,6.135101e+02,2.579291e+02,0.000000,5.950000e+02,5.950000e+02,5.950000e+02,1.900000e+03
scores_research,9969.0,2.609894e+01,1.077100e+01,7.100000,2.460000e+01,2.460000e+01,2.460000e+01,1.000000e+02
scores_research_rank,9969.0,5.474122e+02,2.554737e+02,0.000000,5.185000e+02,5.185000e+02,5.185000e+02,1.890000e+03
scores_citations,9969.0,5.744357e+01,1.448260e+01,3.400000,5.800000e+01,5.800000e+01,5.800000e+01,1.000000e+02
scores_citations_rank,9969.0,6.383907e+02,2.634819e+02,0.000000,6.190000e+02,6.190000e+02,6.190000e+02,1.887000e+03
scores_industry_income,9969.0,4.588717e+01,1.237924e+01,15.600000,4.330000e+01,4.330000e+01,4.330000e+01,1.000000e+02


In [46]:
df1=pd.read_csv("final_master.csv")

In [47]:
df1.isnull().sum()

university_id                0
normalized_name              0
university_name              0
country_name                 0
university                   0
                            ..
tertiary_enrollment_ratio    0
population                   0
literacy_rate_pct            0
data_source                  0
download_timestamp_utc       0
Length: 61, dtype: int64

In [48]:
df1.columns.tolist()

['university_id',
 'normalized_name',
 'university_name',
 'country_name',
 'university',
 'country_qs',
 'year',
 'qs_rank',
 'qs_score',
 'academic_reputation_score',
 'employer_reputation_score',
 'faculty_student_score',
 'citations_per_faculty_score',
 'international_faculty_score',
 'international_students_score',
 'international_research_network_score',
 'employment_outcomes_score',
 'sustainability_score',
 'rank_order',
 'rank',
 'name',
 'scores_overall',
 'scores_overall_rank',
 'scores_teaching',
 'scores_teaching_rank',
 'scores_research',
 'scores_research_rank',
 'scores_citations',
 'scores_citations_rank',
 'scores_industry_income',
 'scores_industry_income_rank',
 'scores_international_outlook',
 'scores_international_outlook_rank',
 'location',
 'stats_number_students',
 'stats_student_staff_ratio',
 'stats_pc_intl_students',
 'stats_female_male_ratio',
 'aliases',
 'subjects_offered',
 'closed',
 'unaccredited',
 'normalized_name_the',
 'stats_proportion_of_isr',
 '

In [49]:
df1.head(n=200)

,university_id,normalized_name,university_name,country_name,university,country_qs,year,qs_rank,qs_score,academic_reputation_score,...,matched_name,score,country_code,gdp_per_capita_usd,education_expenditure_pct_gdp,tertiary_enrollment_ratio,population,literacy_rate_pct,data_source,download_timestamp_utc
0,U000001,aalborg university,Aalborg University,Denmark,Aalborg University,Denmark,2017.0,374,31.5,5.7,...,AKAD University,93.103448,USA,37518.463531,4.890402,74.491310,66740000.0,95.25,World Bank API,2026-07-13 06:56:42 UTC
1,U000001,aalborg university,Aalborg University,Denmark,Aalborg University,Denmark,2018.0,379,32.2,5.7,...,AKAD University,93.103448,USA,37518.463531,4.890402,74.491310,66740000.0,95.25,World Bank API,2026-07-13 06:56:42 UTC
2,U000001,aalborg university,Aalborg University,Denmark,Aalborg University,Denmark,2019.0,343,31.6,5.7,...,AKAD University,93.103448,USA,37518.463531,4.890402,74.491310,66740000.0,95.25,World Bank API,2026-07-13 06:56:42 UTC
3,U000001,aalborg university,Aalborg University,Denmark,Aalborg University,Denmark,2020.0,324,33.0,5.7,...,AKAD University,93.103448,DNK,60985.488560,7.385750,82.664330,5831404.0,95.25,World Bank API,2026-07-13 06:56:42 UTC
4,U000001,aalborg university,Aalborg University,Denmark,Aalborg University,Denmark,2021.0,305,34.0,5.7,...,AKAD University,93.103448,DNK,69340.733492,6.959160,83.982292,5856733.0,95.25,World Bank API,2026-07-13 06:56:42 UTC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,U000093,albaydha university,Albaydha University,Yemen,University of Dundee,United States,2020.0,801-1000,25.2,5.7,...,AKAD University,93.103448,USA,37518.463531,4.890402,74.491310,66740000.0,95.25,World Bank API,2026-07-13 06:56:42 UTC
196,U000094,albert ludwigs universitaet freiburg,Albert-Ludwigs-Universitaet Freiburg,Germany,Albert-Ludwigs-Universitaet Freiburg,Germany,2017.0,163,52.2,5.7,...,AKAD University,93.103448,USA,37518.463531,4.890402,74.491310,66740000.0,95.25,World Bank API,2026-07-13 06:56:42 UTC
197,U000094,albert ludwigs universitaet freiburg,Albert-Ludwigs-Universitaet Freiburg,Germany,Albert-Ludwigs-Universitaet Freiburg,Germany,2018.0,171,50.8,5.7,...,AKAD University,93.103448,USA,37518.463531,4.890402,74.491310,66740000.0,95.25,World Bank API,2026-07-13 06:56:42 UTC
198,U000094,albert ludwigs universitaet freiburg,Albert-Ludwigs-Universitaet Freiburg,Germany,Albert-Ludwigs-Universitaet Freiburg,Germany,2019.0,186,45.4,5.7,...,AKAD University,93.103448,USA,37518.463531,4.890402,74.491310,66740000.0,95.25,World Bank API,2026-07-13 06:56:42 UTC


In [50]:
df1.drop(columns=["university","country_code","download_timestamp_utc","literacy_rate_pct"])

,university_id,normalized_name,university_name,country_name,country_qs,year,qs_rank,qs_score,academic_reputation_score,employer_reputation_score,...,h_index,i10_index,2yr_mean_citedness,matched_name,score,gdp_per_capita_usd,education_expenditure_pct_gdp,tertiary_enrollment_ratio,population,data_source
0,U000001,aalborg university,Aalborg University,Denmark,Denmark,2017.0,374,31.5,5.7,2.5,...,547.0,72656.0,3.121947,AKAD University,93.103448,37518.463531,4.890402,74.491310,66740000.0,World Bank API
1,U000001,aalborg university,Aalborg University,Denmark,Denmark,2018.0,379,32.2,5.7,2.5,...,547.0,72656.0,3.121947,AKAD University,93.103448,37518.463531,4.890402,74.491310,66740000.0,World Bank API
2,U000001,aalborg university,Aalborg University,Denmark,Denmark,2019.0,343,31.6,5.7,2.5,...,547.0,72656.0,3.121947,AKAD University,93.103448,37518.463531,4.890402,74.491310,66740000.0,World Bank API
3,U000001,aalborg university,Aalborg University,Denmark,Denmark,2020.0,324,33.0,5.7,2.5,...,547.0,72656.0,3.121947,AKAD University,93.103448,60985.488560,7.385750,82.664330,5831404.0,World Bank API
4,U000001,aalborg university,Aalborg University,Denmark,Denmark,2021.0,305,34.0,5.7,2.5,...,547.0,72656.0,3.121947,AKAD University,93.103448,69340.733492,6.959160,83.982292,5856733.0,World Bank API
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9964,U003561,zhytomyr polytechnic state university,Zhytomyr Polytechnic State University,Ukraine,United States,2020.0,801-1000,25.2,5.7,2.5,...,41.0,474.0,0.732987,AKAD University,93.103448,37518.463531,4.890402,74.491310,66740000.0,World Bank API
9965,U003562,ziane achour university of djelfa,Ziane Achour University of Djelfa,Algeria,United States,2020.0,801-1000,25.2,5.7,2.5,...,66.0,1115.0,3.158951,AKAD University,93.103448,37518.463531,4.890402,74.491310,66740000.0,World Bank API
9966,U003563,ziauddin university,Ziauddin University,Pakistan,United States,2020.0,801-1000,25.2,5.7,2.5,...,94.0,1770.0,1.041774,AKAD University,93.103448,37518.463531,4.890402,74.491310,66740000.0,World Bank API
9967,U003564,zonguldak bulent ecevit university,Zonguldak Bülent Ecevit University,Turkey,United States,2020.0,801-1000,25.2,5.7,2.5,...,148.0,7450.0,1.670994,AKAD University,93.103448,37518.463531,4.890402,74.491310,66740000.0,World Bank API
